# MMScan data check

Exploratory pass over the freshly-downloaded MMScan / EmbodiedScan-v1 / EmbodiedScan-v2-beta data before writing any real preprocessing.
matterport3d raw scans are **not downloaded yet** -- cells below flag anything that touches it instead of failing.

In [ ]:
import json, pickle, os
from pathlib import Path
import numpy as np

BASE = Path("/glob/g01-cache/pf/Yushuo/vjepa201")
MMSCAN_ROOT = BASE / "source_data/mmscan_data"
SPLIT_DIR = MMSCAN_ROOT / "embodiedscan_split"
V1_DIR = SPLIT_DIR / "embodiedscan-v1"
V2_DIR = SPLIT_DIR / "embodiedscan-v2"
BETA_DIR = MMSCAN_ROOT / "MMScan-beta-release"

RAW_ROOTS = {
    "scannet": BASE / "source_data/scannet",
    "3rscan": BASE / "source_data/3rscan",
    "matterport3d": BASE / "source_data/matterport3d",  # not downloaded yet
}

for name, p in {"mmscan_root": MMSCAN_ROOT, "v1": V1_DIR, "v2": V2_DIR, "beta": BETA_DIR, **RAW_ROOTS}.items():
    print(f"{'OK ' if p.exists() else 'MISSING '} {name:15s} {p}")

## Directory tree (depth-limited)
Just to see what actually landed on disk before assuming any filenames.

In [ ]:
def print_tree(root: Path, max_depth=3, max_items=15):
    if not root.exists():
        print(f"[missing] {root}")
        return
    root_depth = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root):
        dirpath = Path(dirpath)
        depth = len(dirpath.parts) - root_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{dirpath.name}/")
        dirnames.sort()
        for d in dirnames[:max_items]:
            pass  # walked next iteration, just counting here
        if len(dirnames) > max_items:
            print(f"{indent}  ... ({len(dirnames)} subdirs total, showing first {max_items})")
            dirnames[:] = dirnames[:max_items]
        for f in sorted(filenames)[:max_items]:
            size = (dirpath / f).stat().st_size
            print(f"{indent}  {f}  ({size/1e6:.2f} MB)")
        if len(filenames) > max_items:
            print(f"{indent}  ... ({len(filenames)} files total, showing first {max_items})")

print_tree(MMSCAN_ROOT, max_depth=3, max_items=15)

## Generic file inspector
Loads json / pkl / npy files found under a root (capped) and prints type + top-level keys/shape, without assuming exact filenames -- those haven't all been confirmed yet.

In [ ]:
def describe(obj, max_items=10):
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"  dict, {len(keys)} keys: {keys[:max_items]}")
        first_key = keys[0] if keys else None
        if first_key is not None:
            print(f"  obj[{first_key!r}] = {str(obj[first_key])[:300]}")
    elif isinstance(obj, list):
        print(f"  list, {len(obj)} items")
        if obj:
            print(f"  obj[0] = {str(obj[0])[:300]}")
    elif isinstance(obj, np.ndarray):
        print(f"  ndarray shape={obj.shape} dtype={obj.dtype}")
    else:
        print(f"  {type(obj)}: {str(obj)[:300]}")

def inspect_root(root: Path, max_files=8, size_limit_mb=200):
    if not root.exists():
        print(f"[missing] {root}")
        return
    seen = 0
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if seen >= max_files:
                return
            path = Path(dirpath) / f
            size_mb = path.stat().st_size / 1e6
            if size_mb > size_limit_mb:
                print(f"\n[skip, {size_mb:.0f}MB > limit] {path}")
                continue
            try:
                if f.endswith(".json"):
                    print(f"\n[json] {path}")
                    describe(json.load(open(path)))
                elif f.endswith(".pkl"):
                    print(f"\n[pkl] {path}")
                    describe(pickle.load(open(path, "rb")))
                elif f.endswith(".npy"):
                    print(f"\n[npy] {path}")
                    describe(np.load(path, allow_pickle=True))
                else:
                    continue
            except Exception as e:
                print(f"\n[error] {path}: {e}")
            seen += 1

In [ ]:
print("=== embodiedscan-v1 ===")
inspect_root(V1_DIR, max_files=8)

In [ ]:
print("=== embodiedscan-v2 ===")
inspect_root(V2_DIR, max_files=8)

In [ ]:
print("=== MMScan-beta-release ===")
inspect_root(BETA_DIR, max_files=8)

## Known asset: embodiedscan_occupancy (v1)
Already confirmed on disk: `embodiedscan_occupancy/<dataset>/<scene_id>/{occupancy.npy, visible_occupancy.pkl}`.

In [ ]:
occ_root = V1_DIR / "embodiedscan_occupancy"
if occ_root.exists():
    for dataset_dir in sorted(occ_root.iterdir()):
        if not dataset_dir.is_dir():
            continue
        scene_dirs = list(dataset_dir.iterdir())
        print(f"{dataset_dir.name}: {len(scene_dirs)} scenes")
        if scene_dirs:
            sample_scene = scene_dirs[0]
            occ = np.load(sample_scene / "occupancy.npy", allow_pickle=True)
            vis = pickle.load(open(sample_scene / "visible_occupancy.pkl", "rb"))
            print(f"  sample: {sample_scene.name}")
            print(f"  occupancy.npy shape={occ.shape} dtype={occ.dtype}")
            describe(vis)
else:
    print(f"[missing] {occ_root}")

## Scene-ID cross-check: annotation vs raw scans on disk
For each source dataset referenced by the occupancy folders, check how many of those scene IDs actually exist under the raw scan roots downloaded so far.

In [ ]:
def raw_scene_ids(dataset_name):
    root = RAW_ROOTS.get(dataset_name)
    if root is None or not root.exists():
        return None
    scans_dir = root / "scans" if (root / "scans").exists() else root
    return {p.name for p in scans_dir.iterdir() if p.is_dir()}

if occ_root.exists():
    for dataset_dir in sorted(occ_root.iterdir()):
        if not dataset_dir.is_dir():
            continue
        ann_ids = {p.name for p in dataset_dir.iterdir() if p.is_dir()}
        raw_ids = raw_scene_ids(dataset_dir.name)
        if raw_ids is None:
            print(f"{dataset_dir.name}: {len(ann_ids)} annotated scenes, raw data NOT downloaded -- blocked")
            continue
        overlap = ann_ids & raw_ids
        print(f"{dataset_dir.name}: {len(ann_ids)} annotated, {len(raw_ids)} raw on disk, {len(overlap)} overlap "
              f"({len(ann_ids - raw_ids)} annotated scenes missing raw data)")

## Optional: MMScan devkit (only if cloned + installed)
The devkit (`rbler1234/MMScan` repo, or the `mmscan` branch of `InternRobotics/EmbodiedScan`) is a separate install step -- this cell is a no-op until that's done.

In [ ]:
try:
    from mmscan import MMScan
    ds = MMScan(split="val", task="MMScan-VG")
    print(f"MMScan devkit loaded, {len(ds)} val/VG samples")
    print(ds[0])
except ImportError:
    print("mmscan devkit not installed yet -- skip (see data_preparation/README.md in the mmscan branch)")